<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_11_3_tokenizers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>


# T81-558: Anwendungen tiefer neuronaler Netzwerke
**Modul 11: Natürliche Sprachverarbeitung mit Hugging Face**
* Dozent: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Jeff Heaton](https://sites.wustl.edu/jeffheaton/)
* Weitere Informationen finden Sie unter [class website](https://sites.wustl.edu/jeffheaton/t81-558/).

# Modul 11 ​​Material

* Teil 11.1: Einführung in Hugging Face [[Video]](https://www.youtube.com/watch?v=PzuL84ksRuE&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi) [[Video]](https://www.youtube.com/watch?v=PzuL84ksRuE&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi)
* Teil 11.2: Umarmendes Gesicht in Python [[Video]](https://www.youtube.com/watch?v=tkGIF4CFoV4&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi) [[Video]](https://www.youtube.com/watch?v=tkGIF4CFoV4&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi)
* **Teil 11.3: Hugging Face-Tokenizer** [[Video]](https://www.youtube.com/watch?v=Cz2nvfK28eI&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi) [[Video]](https://www.youtube.com/watch?v=Cz2nvfK28eI&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi)
* Teil 11.4: Hugging Face-Datensätze [[Video]](https://www.youtube.com/watch?v=yLlCZLzE2XU&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi) [[Video]](https://www.youtube.com/watch?v=yLlCZLzE2XU&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi)
* Teil 11.5: Training von Hugging-Face-Modellen [[Video]](https://www.youtube.com/watch?v=7YZOik5S3vs&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi) [[Video]](https://www.youtube.com/watch?v=7YZOik5S3vs&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi)


# Google CoLab-Anweisungen

Der folgende Code prüft, ob Google CoLab ausgeführt wird.

In [ ]:
try:
    from google.colab import drive

    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Teil 11.3: Hugging Face Tokenizer

Bei der Tokenisierung geht es darum, den Text in Stücke, sogenannte Token, zu zerlegen und dabei möglicherweise bestimmte Zeichen, wie etwa Satzzeichen, wegzulassen. Überlegen Sie, wie das Programm die folgenden Sätze in Wörter zerlegen könnte.

* Dies ist ein Test.
* Ok, aber was ist damit?
* Ist U.S.A. dasselbe wie USA?
* Welcher Datensatz eignet sich am besten?
* Ich denke, ich werde dies tun – nein, warte, ich werde das tun.

Das umarmende Gesicht enthält Tokenisierer, die diese Sätze in Wörter und Teilwörter aufteilen können. Da Englisch und einige andere Sprachen aus gemeinsamen Wortteilen bestehen, tokenisieren wir Teilwörter. Beispielsweise wird ein Gerundium wie „sleeping“ in „sleep“ und „##ing“ tokenisiert.

Wir beginnen bei Bedarf mit der Installation von Hugging Face.


In [ ]:
# Ausgabe ausblenden
!pip install transformers
!pip install transformers[sentencepiece]

Zuerst erstellen wir einen Hugging Face-Tokenizer. Im Hugging Face-Hub sind mehrere verschiedene Tokenizer verfügbar. Für dieses Beispiel verwenden wir den folgenden Tokenizer:

* Distilbert-Base-unverpackt

Dieser Tokenizer basiert auf BERT und geht von einem englischen Text aus, bei dem die Groß- und Kleinschreibung nicht beachtet wird.

In [ ]:
from transformers import AutoTokenizer

model = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model)

Wir können jetzt einen Beispielsatz tokenisieren.

In [ ]:
encoded = tokenizer("Tokenizing text is easy.")
print(encoded)

Das Ergebnis dieser Tokenisierung enthält zwei Elemente:
* input_ids – Die einzelnen Teilwortindizes, jeder Index identifiziert eindeutig ein Teilwort.
* attention_mask – Welche Werte in *input_ids* sind sinnvoll und keine Füllzeichen.
Dieser Satz hatte keine Auffüllung, daher haben alle Elemente eine Aufmerksamkeitsmaske von „1“. Später werden wir eine Ausgabe mit einer festen Länge anfordern und Auffüllung einführen, die immer eine Aufmerksamkeitsmaske von „0“ hat. Obwohl jeder Tokenizer anders implementiert werden kann, ist die Aufmerksamkeitsmaske eines Tokenizers im Allgemeinen entweder „0“ oder „1“.

Aufgrund von Teilwörtern und speziellen Token stimmt die Anzahl der Token möglicherweise nicht mit der Anzahl der Wörter in der Quellzeichenfolge überein. Wir können die Bedeutung der einzelnen Token sehen, indem wir diese IDs wieder in Zeichenfolgen umwandeln.

In [ ]:
tokenizer.convert_ids_to_tokens(encoded.input_ids)

Wie Sie sehen, gibt es am Anfang und Ende jeder Sequenz zwei spezielle Token. Wir werden bald sehen, wie wir diese speziellen Token einschließen oder ausschließen können. Diese speziellen Token können je nach Tokenizer unterschiedlich sein; jedoch beginnt [CLS] eine Sequenz für diesen Tokenizer und [SEP] beendet eine Sequenz. Sie werden auch sehen, dass das Gerundium „tokening“ in „token“ und „*ing“ aufgeteilt ist.

Für diesen Tokenizer kommen die Sondertoken zwischen 100 und 103 vor. Die meisten Hugging Face-Tokenizer verwenden diesen ungefähren Bereich für Sondertoken. Der Wert Null (0) steht normalerweise für Auffüllung. Mit diesem Befehl können wir alle Sondertoken anzeigen.



In [ ]:
tokenizer.convert_ids_to_tokens([0, 100, 101, 102, 103])

Dieser Tokenizer unterstützt diese gängigen Token:

* \[CLS\] - Sequenzanfang.
* \[SEP\] - Sequenzende.
* \[PAD\] – Polsterung.
* \[UNK\] – Unbekanntes Token.
* \[MASK\] - Mask out tokens for a neural network to predict. Not used in this book, see [MLM paper](https://arxiv.org/abs/2109.01819).

Es ist auch möglich, Listen von Sequenzen zu tokenisieren. Wir können Sequenzen auffüllen und kürzen, um eine Standardlänge zu erreichen, indem wir viele Sequenzen auf einmal tokenisieren.



In [ ]:
text = ["This movie was great!", "I hated this move, waste of time!", "Epic?"]

encoded = tokenizer(text, padding=True, add_special_tokens=True)

print("**Input IDs**")
for a in encoded.input_ids:
    print(a)

print("**Attention Mask**")
for a in encoded.attention_mask:
    print(a)

Beachten Sie die **input_id** für die drei Filmkritik-Textsequenzen. Jede dieser Sequenzen beginnt mit 101 und wir füllen mit Nullen auf. Direkt vor dem Auffüllen endet jede ID-Gruppe mit 102. Die Aufmerksamkeitsmasken haben ebenfalls Nullen für jeden der Auffülleinträge.

Wir haben zwei Parameter für den Tokenizer verwendet, um den Tokenisierungsprozess zu steuern. Einige andere nützliche [parameters](https://huggingface.co/docs/transformers/main_classes/tokenizer) sind:

* add_special_tokens (standardmäßig True) Ob die Sequenzen mit den speziellen Token relativ zu ihrem Modell codiert werden sollen oder nicht.
* padding (Standardwert: „False“) Aktiviert und steuert die Kürzung.
* max_length (optional) Steuert die maximale Länge, die von einem der Truncation-/Padding-Parameter verwendet werden soll.